In [ ]:
"""
The Goal of this notebook:
- add a second camera to the satellite
-- different mount (location, orientation) /FOV
-- 

Verification:
- show static rendering of satellite (visibility cone)
--verify mount is correct (visual, forward looking)
--verify changing angle works without fucking up sim
- show classification view array in simulation (video)
-- test take

API:
- go through SimConfig (can be consumed by Sim Kernel)
-- define what is needed (keep running in sim kernel in mind, should tie in SimConfig, can be extended, can be passed as function)
-- (instead of passing target area, where target area is currently parsed send this function)


use defaults:
different color than main camera
(I think we have defined the default already in the first notebook for FOV
and mount point of the camera -> Look up)

sampling_time: 200 microseconds
-> simplification, the time of the image taken is instant,
use closest timestep after cmd release to get current quality

"""

In [ ]:
# Secondary forward-looking camera (GoPro 4K+ mental model) — s01 default: 25° prograde tilt
# Tilt = 25° forward along-track (§A): positive tilt = prograde, boresight shifted ahead of nadir.
# build_setup() sets this automatically; this cell just displays the optics.
from environment_definition.constants.SATELLITE import CAMERA_ALTITUDE
from environment_definition.constants.UNIT_REGISTRY import UREG as ureg
from simulation.camera_image import CameraImage, CameraMount

TILT_ANGLE_SCND = 25 * ureg.deg  # s01 default: 25° forward-looking (§A)
FOV_ANGLE_SCND = 60 * ureg.deg
N_PIXELS_SCND_X = 5312
N_PIXELS_SCND_Y = 2988
PIXEL_SIZE_SCND = 1.55 * ureg.um  # educated guess; TBD from datasheet

scnd_camera = CameraImage.from_fov(
    fov_y=FOV_ANGLE_SCND,
    pixel_size=PIXEL_SIZE_SCND,
    n_pixels_x=N_PIXELS_SCND_X,
    n_pixels_y=N_PIXELS_SCND_Y,
    axis="x"
)
scnd_mount = CameraMount(camera=scnd_camera, tilt_off_nadir=TILT_ANGLE_SCND)

ref_altitude = SATELLITE_ALTITUDE if "SATELLITE_ALTITUDE" in globals() else CAMERA_ALTITUDE
gsd = scnd_camera.gsd_at(ref_altitude)
swath_y = scnd_camera.swath_at(ref_altitude, axis="y")
swath_x = scnd_camera.swath_at(ref_altitude, axis="x")

print("Secondary camera (GoPro-style)")
print(f"  mount tilt off nadir: {scnd_mount.tilt_off_nadir.to('deg'):~}")
print(f"  focal length:         {scnd_camera.focal_length.to('mm'):~}")
print(f"  pixel pitch:          {scnd_camera.pixel_size.to('um'):~}")
print(f"  resolution:           {scnd_camera.n_pixels_x} x {scnd_camera.n_pixels_y}")
print(f"  FOV (y / x):          {scnd_camera.fov(axis='y').to('deg'):~} / {scnd_camera.fov(axis='x').to('deg'):~}")
print(f"  GSD @ {ref_altitude.to('km'):~}:           {gsd.to('m'):~}")
print(f"  swath (y / x):        {swath_y.to('km'):~} / {swath_x.to('km'):~}")


In [ ]:
def image_quality(self, **sim_state: CurrentState)  -> float:
    sat_speed = sim_state["sat_speed"]
    sat_omega = sim_state["sat_omega"]
    distance = sim_state["distance_to_camera_bore_collision"]
    
    

    def _bore_speed(smthing):
        #TODO
        raise NotImplementedError

    def _image_quality(ground_speed, bore_speed, epsilon:float = 1e-3):
        
        if epsilon != 0 and if epsilon < 1:
            return(
                1/
                abs(ground_speed - bore_speed)
                + epsilon
                )
        else: raise ValueError("epsilon is a numeric stabilizer, dont set 0")
    
    gs = sat_speed
    bs = _bore_speed()
    return _image_quality(gs, bs)
